# Ch 4. OpenAI / Anthropic API 시작하기

원본: [AI Assistant Engineering - Part 2 Ch 4](https://desty.github.io/study-ai-assistant-engineering/part2/04-api-start/)

## 이 챕터에서 배우는 것

- **API 호출 = HTTPS로 멀리 있는 모델을 부르는 일** 이라는 직관
- **SDK**(Python 라이브러리)로 Anthropic · OpenAI 첫 10줄 호출
- 메시지의 **세 가지 역할** (`system` · `user` · `assistant`) 이 왜 분리되어 있나
- **핵심 파라미터 4개** (`model` · `max_tokens` · `temperature` · `stop_sequences`) 의 감각
- **에러 · 재시도 · 타임아웃 · 비용** — 한 줄짜리 실험을 운영형 코드로 올리는 최소 습관

In [ ]:
# 필수 패키지 설치
!pip install -q anthropic tenacity

In [ ]:
# API 키 설정
import os
from getpass import getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')

## 4. 최소 예제 — 10줄

In [ ]:
from anthropic import Anthropic

client = Anthropic()  # (1)!

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    messages=[{"role": "user", "content": "LLM API를 한 문장으로 설명해줘"}],
)

print(response.content[0].text)

## 5. 실전 튜토리얼

### 5.1 메시지 배열 — 세 역할

In [ ]:
history = [
    {"role": "user",      "content": "안녕, 내 이름은 desty야."},
    {"role": "assistant", "content": "반가워요, desty님!"},
    {"role": "user",      "content": "내 이름이 뭐였지?"},  # (1)!
]

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=64,
    system="당신은 친절한 한국어 도우미입니다.",
    messages=history,
)
print(response.content[0].text)

### 5.2 핵심 파라미터 4개

In [ ]:
response = client.messages.create(
    model="claude-opus-4-1-20250805",        # 1. 어느 모델을 쓸지
    max_tokens=256,                 # 2. 응답의 최대 길이 (토큰)
    temperature=0.3,                # 3. 0~1.0 — 창의성 다이얼
    stop_sequences=["\n\n---\n\n"], # 4. 이 문자열 나오면 즉시 중단
    system="당신은 전문가입니다.",
    messages=[{"role": "user", "content": "파이썬 프로그래밍의 핵심 개념을 설명해줘"}],
)

print(response.content[0].text)

### 5.3 응답 객체 해부

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    messages=[{"role": "user", "content": "안녕하세요"}],
)

print(f"텍스트: {response.content[0].text}")
print(f"타입: {response.content[0].type}")
print(f"중단 이유: {response.stop_reason}")
print(f"입력 토큰: {response.usage.input_tokens}")
print(f"출력 토큰: {response.usage.output_tokens}")
print(f"응답 모델: {response.model}")

### 5.3 비용 계산

In [ ]:
# 2026-04 기준 대략 단가 (공식 사이트에서 최신 확인)
PRICE_PER_M_INPUT  = {"opus": 15.0, "sonnet": 3.0, "haiku": 0.25}  # 달러 / 100만 토큰
PRICE_PER_M_OUTPUT = {"opus": 75.0, "sonnet": 15.0, "haiku": 1.25}

def estimate_cost(resp, tier="haiku") -> float:
    ip = resp.usage.input_tokens
    op = resp.usage.output_tokens
    return (ip * PRICE_PER_M_INPUT[tier] + op * PRICE_PER_M_OUTPUT[tier]) / 1_000_000

print(f"${estimate_cost(response, 'haiku'):.6f}")

### 5.4 에러 · 재시도 · 타임아웃

In [ ]:
from anthropic import Anthropic, APIStatusError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=16),  # 1s, 2s, 4s... 최대 16s
    retry=retry_if_exception_type(APIStatusError),
)
def ask(prompt: str, model: str = "claude-haiku-4-5-20251001") -> str:
    client = Anthropic(timeout=30.0)  # (1)!
    r = client.messages.create(
        model=model,
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text

# 테스트
result = ask("안녕하세요")
print(result)

## 6. 자주 깨지는 포인트

### 실수 1. API 키를 코드에 하드코딩
- 하지 말 것: `client = Anthropic(api_key="sk-ant-xxxxx")`
- 대신: 환경변수, .env, Colab Secrets, AWS Secrets Manager

### 실수 2. `max_tokens` 를 입력 상한으로 착각
- `max_tokens`는 **출력** 길이 상한
- 입력(프롬프트)은 모델의 컨텍스트 윈도우에 따름
- 예상 출력의 1.5~2배로 설정

### 실수 3. 타임아웃·재시도 없이 호출
- 네트워크 이슈 한 번에 프로그램이 죽음
- 항상 `timeout=30` + `tenacity` 래퍼 사용

### 실수 4. `assistant` 역할에 내 생각을 박기
- 모델이 이미 말한 것처럼 취급됨
- 지시는 `system`에, 예시는 few-shot 패턴으로

### 실수 5. Rate limit 도달 시 무한 재시도 루프
- 재시도 상한 필수 (3~5회)
- 지수 backoff 사용

## 7. 운영 시 체크할 점

- [ ] **API 키** 환경변수 또는 시크릿 매니저
- [ ] **비용 상한** — Anthropic 콘솔에서 월 한도 설정
- [ ] **타임아웃** — 30초 이내
- [ ] **재시도** — `tenacity` 3~5회 exponential backoff
- [ ] **모델 버전 pin** — 예: `"claude-haiku-4-5-20251001"`
- [ ] **토큰 · 비용 로깅** — 매 호출 usage 기록
- [ ] **Latency 측정** — p50 / p95 / p99 추적
- [ ] **PII 마스킹** — 전송 전 개인정보 제거·익명화

---

**다음 챕터** → Ch 5. 프롬프트 엔지니어링 + CoT 기초

지금은 질문 하나 던지고 답을 받았지만, system 프롬프트를 잘 쓰면 모델이 완전히 다른 어시스턴트가 됩니다.